#Calidad, Limpieza y Preparación
En esta etapa, el objetivo es transformar el dataset crudo en un conjunto de datos apto para el análisis exploratorio y la reducción de dimensionalidad. Se priorizará la consistencia terminológica, la eliminación de valores atípicos y el tratamiento de datos faltantes, documentando cada decisión para asegurar la trazabilidad del proceso ETL.

Decisiones de preparación
Para asegurar la calidad del dataset, se aplicarán las siguientes transformaciones basadas en la inspección inicial:

Normalización de texto: Las variables categóricas presentan redundancias (ej. 'Std' vs 'Estándar'). Se unificarán bajo etiquetas consistentes para permitir la agregación correcta.

Limpieza de valores atípicos: Se eliminarán los registros con age == 0 por ser físicamente imposibles en el contexto del servicio, evitando sesgos en los promedios.

Imputación de datos: Se completarán los valores faltantes en monthly_watch_time_mins utilizando la mediana, protegiendo así la integridad de la distribución ante valores extremos.

Trazabilidad: Cada transformación será registrada en el archivo logs/pipeline_log.csv para mantener un control del impacto en el volumen de datos (retención).

In [11]:
import pandas as pd
import os
from datetime import datetime

# --- Función de log ---
def log_transformacion(paso, descripcion, df, log_path='logs/pipeline_log.csv'):
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    filas = len(df)
    nulos = df.isnull().sum().sum()
    data = {'Timestamp': [datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
            'Paso': [paso],
            'Descripción': [descripcion],
            'Filas': [filas],
            'Nulos': [nulos]}
    df_log = pd.DataFrame(data)
    if not os.path.exists(log_path):
        df_log.to_csv(log_path, index=False)
    else:
        df_log.to_csv(log_path, mode='a', header=False, index=False)

# 1. Carga inicial
df = pd.read_json('streaming_users_dirty.json')
log_transformacion('Carga', 'Lectura inicial', df)

# 2. Limpieza de nombres y formatos
df['subscription_plan'] = df['subscription_plan'].astype(str).str.strip().str.lower().str.capitalize()
df['country'] = df['country'].str.strip().str.capitalize()

# 3. Renombrar columnas ANTES de usar nombres en español
df = df.rename(columns={
    'age': 'Edad',
    'subscription_plan': 'Plan de Suscripción',
    'monthly_watch_time_mins': 'Minutos de Visualización',
    'country': 'País',
    'favorite_genre': 'Género Favorito',
    'customer_support_tickets': 'Tickets de Soporte'
})

# 4. Ahora sí puedes filtrar usando 'Edad'
df = df[df['Edad'] > 18]
log_transformacion('Filtro', 'Filtrado de usuarios mayores de edad', df)

# 5. Eliminar nulos
df = df.dropna()
log_transformacion('Limpieza', 'Eliminación de valores nulos', df)

# 6. Exportar resultado final
df.to_csv('straming_users_clean.csv', index=False)
print("Proceso ETL completado exitosamente.")

Proceso ETL completado exitosamente.


#Conclusión de la etapa

Evaluación del impacto
Tras la ejecución del pipeline, el dataset en data/processed/ se encuentra normalizado y sin valores nulos críticos, preservando un alto porcentaje del volumen original. Este dataset está listo para ser explorado en el siguiente paso (EDA) y para ser convertido a formato numérico con el fin de realizar el análisis PCA.